# EXP-07: P1 Residue Energetic Contribution
**Per-residue energy decomposition on BPTI–Trypsin (EXP-04 trajectory)**

- **Hypothesis:** P1 (K15) is #1 ranked contributor, >40% of total binding energy
- **PASS:** P1 rank #1 AND >40% | **MARGINAL:** top-3 AND 30–40% | **FAIL:** otherwise
- **Depends on:** EXP-04 production trajectory + system.xml on Drive

In [ ]:
# Install OpenMM + all scientific dependencies (pip — OpenCL platform)
!pip install openmm mdtraj 'pymbar==4.0.3' mdanalysis
!pip install netCDF4 git+https://github.com/choderalab/openmmtools.git

# mpiplus stub — not on PyPI; minimal shim for imports
import os as _os, sys as _sys
_sp = _os.path.join(_sys.prefix, 'lib',
    f'python{_sys.version_info.major}.{_sys.version_info.minor}',
    'site-packages', 'mpiplus')
_os.makedirs(_sp, exist_ok=True)
with open(_os.path.join(_sp, '__init__.py'), 'w') as _f:
    _f.write(
        'class delayed_termination:\n'
        '    def __init__(self, func=None):\n'
        '        self._func = func\n'
        '    def __enter__(self): return self\n'
        '    def __exit__(self, *a): pass\n'
        '    def __call__(self, *args, **kwargs):\n'
        '        if self._func is not None: return self._func(*args, **kwargs)\n'
        '        if len(args) == 1 and callable(args[0]): return args[0]\n'
        '        return self\n'
        'def get_mpicomm(): return None\n'
        'def run_single_node(rank=0, broadcast_result=False):\n'
        '    def decorator(func): return func\n'
        '    return decorator\n'
        'def on_single_node(rank=0, broadcast_result=False, sync_nodes=False):\n'
        '    def decorator(func): return func\n'
        '    return decorator\n'
        'def distribute(func, seq, *args, send_results_to=None, sync_nodes=False, group_nodes=None):\n'
        '    results = [func(item, *args) for item in seq]\n'
        '    return list(zip(*results)) if results and isinstance(results[0], tuple) else (results, list(seq))\n'
    )
for _k in list(_sys.modules):
    if 'mpiplus' in _k:
        del _sys.modules[_k]

!pip install -q scipy matplotlib pandas requests gemmi

# Verify critical imports
import importlib
all_ok = True
for pkg in ['openmm', 'openmmtools', 'mdtraj', 'pymbar', 'MDAnalysis']:
    try:
        m = importlib.import_module(pkg)
        print(f"\u2713 {pkg} {getattr(m, '__version__', '')}")
    except ImportError as e:
        print(f"\u2717 {pkg}: {e}")
        all_ok = False

if all_ok:
    print("\n\u2705 All packages installed successfully!")
else:
    raise RuntimeError("Package installation failed \u2014 check error messages above")

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive', force_remount=True)
import sys, os, shutil, json, zipfile, logging
from pathlib import Path
import numpy as np

PIPELINE_ROOT = Path('/content/drive/MyDrive/spink7_pipeline')
assert PIPELINE_ROOT.exists(), 'Upload src/ to Drive/MyDrive/spink7_pipeline/'
sys.path.insert(0, str(PIPELINE_ROOT))

EXP_ID = 'EXP-07'
DRIVE_OUTPUT = Path(f'/content/drive/MyDrive/v3_gpu_results/{EXP_ID}')
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
WORK_DIR = Path(f'/content/{EXP_ID}_work')
for d in ['analysis','figures']: (WORK_DIR/d).mkdir(parents=True, exist_ok=True)

EXP04 = Path('/content/drive/MyDrive/v3_gpu_results/EXP-04')
assert EXP04.exists(), 'Run EXP-04 first'
logging.basicConfig(level=logging.INFO)

# ── Console log capture ──────────────────────────────────────
import io as _io
_log_buf = _io.StringIO()
_orig_stdout_write = sys.stdout.write
_orig_stderr_write = sys.stderr.write
def _stdout_log_write(data):
    _orig_stdout_write(data)
    _log_buf.write(data)
def _stderr_log_write(data):
    _orig_stderr_write(data)
    _log_buf.write(data)
sys.stdout.write = _stdout_log_write
sys.stderr.write = _stderr_log_write
# ─────────────────────────────────────────────────────────────

In [ ]:
import openmm
from openmm import unit, Platform
Platform.getPlatformByName('CUDA')
print(f'OpenMM {openmm.__version__}, CUDA ready')

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import mdtraj as md
from openmm import XmlSerializer
from openmm.app import PDBFile, Simulation
from openmm import LangevinMiddleIntegrator
from src.simulate.platform import select_platform
from src.config import KCAL_TO_KJ
print('Imports OK')

In [ ]:
# Load EXP-04 production trajectory (last 50 ns)
top_pdb = str(list(EXP04.glob('**/topology_reference.pdb'))[0])
traj_files = sorted(EXP04.glob('**/production*.dcd'))
if not traj_files:
    traj_files = sorted(EXP04.glob('**/production*.xtc'))
assert traj_files, 'No production trajectory found in EXP-04'
traj = md.load(str(traj_files[0]), top=top_pdb, stride=10)
n_total = traj.n_frames
# Use last 50 ns worth of frames
frames_50ns = min(n_total, 500)
traj_analysis = traj[-frames_50ns:]
print(f'Analysis: {traj_analysis.n_frames} frames from last portion')

In [ ]:
# Identify chains and P1 residue
topology = traj_analysis.topology
chains = list(topology.chains)
print(f'Chains: {len(chains)}')

# In 2PTC: chain 0 = trypsin (E), chain 1 = BPTI (I)
# Identify by size — larger chain is trypsin
chain_sizes = [(c.index, sum(1 for a in c.atoms)) for c in chains if c.n_residues > 5]
chain_sizes.sort(key=lambda x: -x[1])
trypsin_chain_idx = chain_sizes[0][0]
bpti_chain_idx = chain_sizes[1][0]

bpti_residues = [r for r in topology.residues if r.chain.index == bpti_chain_idx]
trypsin_atoms = topology.select(f'chainid {trypsin_chain_idx}')
bpti_atoms = topology.select(f'chainid {bpti_chain_idx}')

# P1 = K15 (Lys15 in BPTI)
p1_residue = None
for r in bpti_residues:
    if r.resSeq == 15:
        p1_residue = r
        break
assert p1_residue is not None, 'K15 not found'
p1_key = f'{p1_residue.name}{p1_residue.resSeq}'
print(f'P1: {p1_key}, BPTI: {len(bpti_residues)} res, Trypsin: {len(trypsin_atoms)} atoms')

In [ ]:
# Per-residue interaction energy: Coulombic approximation via distance-based scoring
# For each BPTI residue, compute contact-weighted interaction with trypsin
trypsin_idx_set = set(trypsin_atoms.tolist())
sample_traj = traj_analysis[::5]  # subsample for speed

per_residue_energies = {}
for res in bpti_residues:
    res_atoms = np.array([a.index for a in res.atoms])
    # Build pairs with nearby trypsin atoms
    pairs = np.array([[ra, ta] for ra in res_atoms for ta in trypsin_atoms[:200]])
    if len(pairs) == 0:
        per_residue_energies[f'{res.name}{res.resSeq}'] = 0.0
        continue
    # Compute distances across sampled frames
    distances = md.compute_distances(sample_traj, pairs)  # (n_frames, n_pairs)
    # Contact score: sum of 1/r^6 for contacts < 1.0 nm (LJ-like proxy)
    contact_mask = distances < 1.0
    scores = np.where(contact_mask, -1.0 / (distances + 0.01)**2, 0.0)
    mean_score = np.mean(np.sum(scores, axis=1))
    per_residue_energies[f'{res.name}{res.resSeq}'] = mean_score

# Rank residues (most stabilizing = most negative)
sorted_residues = sorted(per_residue_energies.items(), key=lambda x: x[1])
print('\nTop 10 contributing residues:')
for i, (name, energy) in enumerate(sorted_residues[:10]):
    print(f'  {i+1}. {name}: {energy:.3f}')

p1_energy = per_residue_energies.get(p1_key, 0)
total_energy = sum(per_residue_energies.values())
p1_fraction = abs(p1_energy) / abs(total_energy) * 100 if total_energy != 0 else 0
p1_rank = [name for name, _ in sorted_residues].index(p1_key) + 1
print(f'\nP1 ({p1_key}): rank #{p1_rank}, fraction = {p1_fraction:.1f}%')

In [ ]:
# Per-residue BSA decomposition
# Total BSA = SASA(A) + SASA(B) - SASA(AB)
sasa_complex = md.shrake_rupley(traj_analysis[::10], mode='residue')
sasa_bpti = md.shrake_rupley(traj_analysis[::10].atom_slice(bpti_atoms), mode='residue')

# Per-residue BSA for BPTI residues
per_residue_bsa = {}
for i, res in enumerate(bpti_residues):
    key = f'{res.name}{res.resSeq}'
    # BSA = SASA_free - SASA_complex for this residue
    sasa_free_mean = np.mean(sasa_bpti[:, i]) if i < sasa_bpti.shape[1] else 0
    # Find corresponding residue index in complex
    res_idx_complex = res.index
    sasa_complex_mean = np.mean(sasa_complex[:, res_idx_complex]) if res_idx_complex < sasa_complex.shape[1] else 0
    per_residue_bsa[key] = max(0, sasa_free_mean - sasa_complex_mean)

total_bsa = sum(per_residue_bsa.values())
p1_bsa = per_residue_bsa.get(p1_key, 0)
p1_bsa_fraction = p1_bsa / total_bsa * 100 if total_bsa > 0 else 0
print(f'Total BSA: {total_bsa:.4f} nm\u00b2')
print(f'P1 BSA fraction: {p1_bsa_fraction:.1f}%')

In [ ]:
# Classification
if p1_rank == 1 and p1_fraction > 40:
    verdict = 'PASS'
elif p1_rank <= 3 and p1_fraction > 30:
    verdict = 'MARGINAL'
else:
    verdict = 'FAIL'
print(f'EXP-07 verdict: {verdict}')

In [ ]:
# Figures
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Fig 1: Per-residue energy bar chart
ax = axes[0]
top15 = sorted_residues[:15]
names = [n for n,_ in top15]; energies = [e for _,e in top15]
colors = ['red' if p1_key in n else 'steelblue' for n in names]
ax.bar(range(len(names)), energies, color=colors, edgecolor='black')
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Interaction Score'); ax.set_title('Top 15 Per-Residue Contributions')

# Fig 2: BSA decomposition
ax = axes[1]
sorted_bsa = sorted(per_residue_bsa.items(), key=lambda x: -x[1])[:15]
bnames = [n for n,_ in sorted_bsa]; bvals = [v for _,v in sorted_bsa]
bcolors = ['red' if p1_key in n else 'gold' for n in bnames]
ax.bar(range(len(bnames)), bvals, color=bcolors, edgecolor='black')
ax.set_xticks(range(len(bnames))); ax.set_xticklabels(bnames, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('BSA (nm\u00b2)'); ax.set_title('Per-Residue BSA')

# Fig 3: P1 dominance pie
ax = axes[2]
ax.pie([p1_fraction, 100-p1_fraction], labels=[f'P1 ({p1_key})', 'Other'],
       colors=['red', 'lightgray'], autopct='%1.1f%%', startangle=90)
ax.set_title(f'P1 Energy Fraction \u2014 {verdict}')

fig.suptitle('EXP-07: P1 Residue Energetic Contribution', fontsize=14)
fig.tight_layout(); fig.savefig(WORK_DIR/'figures'/'results.png', dpi=150); plt.close(fig)
print('Figures saved')

In [ ]:
results = {
    'experiment_id': EXP_ID, 'p1_residue': p1_key,
    'p1_rank': p1_rank, 'p1_energy_fraction_pct': float(p1_fraction),
    'p1_bsa_fraction_pct': float(p1_bsa_fraction),
    'top_5': [{'name': n, 'score': float(e)} for n,e in sorted_residues[:5]],
    'verdict': verdict,
}
with open(WORK_DIR/'results.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)

for item in WORK_DIR.rglob('*'):
    if item.is_file():
        dest = DRIVE_OUTPUT / item.relative_to(WORK_DIR)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(item, dest)
# ── Save full console log ────────────────────────────────────
_console_log_text = _log_buf.getvalue()
(WORK_DIR / 'console_log.txt').write_text(_console_log_text)
(DRIVE_OUTPUT / 'console_log.txt').write_text(_console_log_text)
print(f'Console log saved: {len(_console_log_text)} chars')
# ─────────────────────────────────────────────────────────────
zip_path = Path(f'/content/{EXP_ID}_results.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for item in WORK_DIR.rglob('*'):
        if item.is_file(): zf.write(item, f'{EXP_ID}/{item.relative_to(WORK_DIR)}')
print(f'Zip: {zip_path.stat().st_size/1e6:.1f} MB')
files.download(str(zip_path))